# Linear Collector Simulation

จำลองการทำงานของ `LinearCollector` และ `Simplifier`  
จาก `src/algebra/linear/linear_collector.hpp` และ `simplify.hpp`

## อัลกอริธึม: LinearCollector (AST Visitor)

`LinearCollector` เดิน AST recursively และสร้าง `LinearForm`  
ซึ่งแทน affine expression: `∑ coeffs[xᵢ]·xᵢ + constant`

### Visit rules (pattern matching บน node type)
```
visit(Number n)     → LinearForm(coeffs={}, constant=n.value)
visit(Variable v)   → LinearForm(coeffs={v:1}, constant=0)        ถ้าไม่มีใน context
                   → collect(context[v])                            ถ้ามีใน context
visit(UnaryOp -)    → -1 × collect(child)
visit(BinaryOp +)   → collect(lhs) + collect(rhs)
visit(BinaryOp -)   → collect(lhs) - collect(rhs)
visit(BinaryOp *)   → ถ้าฝั่งใดฝั่งหนึ่งเป็น constant → scalar × linear
                       ถ้าทั้งสองมี variable → ERROR (non-linear)
visit(BinaryOp /)   → ถ้า rhs เป็น constant → lhs × (1/rhs)
                       ถ้า rhs มี variable → ERROR
visit(BinaryOp ^)   → ถ้า exponent=1 → ผ่าน; ถ้า base constant → ค่าตัวเลข
                       ถ้า variable^n, n≠1 → ERROR (non-linear)
visit(FunctionCall) → ถ้า arg constant → evaluate numerically
                       ถ้า arg มี variable → ERROR
```

### LinearForm arithmetic
```
LinearForm + LinearForm → merge coeffs (sum per variable), sum constants
LinearForm * scalar     → scale all coeffs and constant
LinearForm - LinearForm → (form1) + (-1 × form2)
simplify(epsilon)       → prune |coeff| < epsilon
```

In [ ]:
from __future__ import annotations
import math
from dataclasses import dataclass, field
from typing import Any


# ─────────────────────────────────────
# AST Node types (simplified)
# ─────────────────────────────────────

@dataclass
class NumberExpr:
    value: float
    def __repr__(self): return str(self.value)

@dataclass
class VariableExpr:
    name: str
    def __repr__(self): return self.name

@dataclass
class UnaryExpr:
    op: str      # "-"
    operand: Any
    def __repr__(self): return f"({self.op}{self.operand})"

@dataclass
class BinaryExpr:
    op: str      # "+", "-", "*", "/", "^"
    lhs: Any
    rhs: Any
    def __repr__(self): return f"({self.lhs} {self.op} {self.rhs})"

@dataclass
class CallExpr:
    name: str
    arg: Any
    def __repr__(self): return f"{self.name}({self.arg})"


# ─────────────────────────────────────
# LinearForm
# ─────────────────────────────────────

@dataclass
class LinearForm:
    coeffs: dict[str, float] = field(default_factory=dict)
    constant: float = 0.0

    def get_coeff(self, var: str) -> float:
        return self.coeffs.get(var, 0.0)

    def variables(self) -> list[str]:
        """ตัวแปรที่มี coeff ≠ 0 (เรียงตัวอักษร)"""
        return sorted(v for v, c in self.coeffs.items() if abs(c) > 1e-12)

    def is_constant(self) -> bool:
        return len(self.variables()) == 0

    def simplify(self, epsilon: float = 1e-12) -> "LinearForm":
        new_coeffs = {v: c for v, c in self.coeffs.items() if abs(c) > epsilon}
        return LinearForm(new_coeffs, self.constant)

    def __add__(self, other: "LinearForm") -> "LinearForm":
        result = dict(self.coeffs)
        for v, c in other.coeffs.items():
            result[v] = result.get(v, 0.0) + c
        return LinearForm(result, self.constant + other.constant)

    def __sub__(self, other: "LinearForm") -> "LinearForm":
        return self + (other * -1.0)

    def __mul__(self, scalar: float) -> "LinearForm":
        new_coeffs = {v: c * scalar for v, c in self.coeffs.items()}
        return LinearForm(new_coeffs, self.constant * scalar)

    def __repr__(self) -> str:
        terms = [f"{c:+.4g}·{v}" for v, c in sorted(self.coeffs.items())]
        return f"LinearForm({' '.join(terms)} {self.constant:+.4g})"


class NonLinearError(Exception):
    pass


print("AST nodes and LinearForm loaded.")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# LinearCollector
# เดิน AST recursively ด้วย pattern matching บน node type
# ─────────────────────────────────────────────────────────────────

KNOWN_FUNCS = {
    "sin": math.sin, "cos": math.cos, "tan": math.tan,
    "sqrt": math.sqrt, "log": math.log, "exp": math.exp,
    "abs": abs,
}


def collect(
    expr: Any,
    context: dict[str, Any] | None = None,
    depth: int = 0,
    verbose: bool = True,
) -> LinearForm:
    """
    LinearCollector.collect(expr) → LinearForm

    Parameters
    ----------
    expr : AST node
    context : dict[str, expr]  — variable substitutions
    depth : int — recursion depth (สำหรับ indentation)
    verbose : bool — แสดง trace

    Raises
    ------
    NonLinearError  ถ้า expression ไม่ใช่ linear
    """
    indent = "  " * depth
    ctx = context or {}

    if verbose:
        print(f"{indent}collect({expr!r})")

    # ── Number ──────────────────────────────────────────────────
    if isinstance(expr, NumberExpr):
        result = LinearForm(constant=expr.value)
        if verbose:
            print(f"{indent}  → Number({expr.value}) → {result}")
        return result

    # ── Variable ─────────────────────────────────────────────────
    if isinstance(expr, VariableExpr):
        if expr.name in ctx:
            if verbose:
                print(f"{indent}  → Variable '{expr.name}' found in context, substitute")
            # lazy substitution: recurse into context value
            return collect(ctx[expr.name], ctx, depth + 1, verbose)
        else:
            result = LinearForm(coeffs={expr.name: 1.0})
            if verbose:
                print(f"{indent}  → Variable '{expr.name}' (unknown) → {result}")
            return result

    # ── UnaryOp ──────────────────────────────────────────────────
    if isinstance(expr, UnaryExpr):
        if expr.op == "-":
            child = collect(expr.operand, ctx, depth + 1, verbose)
            result = child * -1.0
            if verbose:
                print(f"{indent}  → Negate → {result}")
            return result
        raise NonLinearError(f"Unknown unary op: {expr.op}")

    # ── BinaryOp ─────────────────────────────────────────────────
    if isinstance(expr, BinaryExpr):
        op = expr.op

        if op == "+":
            lhs = collect(expr.lhs, ctx, depth + 1, verbose)
            rhs = collect(expr.rhs, ctx, depth + 1, verbose)
            result = lhs + rhs
            if verbose:
                print(f"{indent}  → Add: {lhs} + {rhs} → {result}")
            return result

        if op == "-":
            lhs = collect(expr.lhs, ctx, depth + 1, verbose)
            rhs = collect(expr.rhs, ctx, depth + 1, verbose)
            result = lhs - rhs
            if verbose:
                print(f"{indent}  → Sub: {lhs} - {rhs} → {result}")
            return result

        if op == "*":
            lhs = collect(expr.lhs, ctx, depth + 1, verbose)
            rhs = collect(expr.rhs, ctx, depth + 1, verbose)
            if verbose:
                print(f"{indent}  → Mul: lhs.constant={lhs.is_constant()}, rhs.constant={rhs.is_constant()}")
            if lhs.is_constant():
                result = rhs * lhs.constant
            elif rhs.is_constant():
                result = lhs * rhs.constant
            else:
                raise NonLinearError(f"Non-linear: variable × variable in {expr}")
            if verbose:
                print(f"{indent}  → Mul result → {result}")
            return result

        if op == "/":
            lhs = collect(expr.lhs, ctx, depth + 1, verbose)
            rhs = collect(expr.rhs, ctx, depth + 1, verbose)
            if not rhs.is_constant():
                raise NonLinearError(f"Non-linear: variable in denominator in {expr}")
            if abs(rhs.constant) < 1e-300:
                raise NonLinearError(f"Division by zero in {expr}")
            result = lhs * (1.0 / rhs.constant)
            if verbose:
                print(f"{indent}  → Div by {rhs.constant} → {result}")
            return result

        if op == "^":
            lhs = collect(expr.lhs, ctx, depth + 1, verbose)
            rhs = collect(expr.rhs, ctx, depth + 1, verbose)
            if not rhs.is_constant():
                raise NonLinearError(f"Non-linear: variable exponent in {expr}")
            exp = rhs.constant
            if verbose:
                print(f"{indent}  → Pow: exponent={exp}, base_constant={lhs.is_constant()}")
            if lhs.is_constant():
                # constant^constant → evaluate
                result = LinearForm(constant=lhs.constant ** exp)
            elif abs(exp - 1.0) < 1e-9:
                result = lhs   # x^1 = x
            else:
                raise NonLinearError(f"Non-linear: variable^{exp} in {expr}")
            if verbose:
                print(f"{indent}  → Pow result → {result}")
            return result

        raise NonLinearError(f"Unknown binary op: {op}")

    # ── FunctionCall ─────────────────────────────────────────────
    if isinstance(expr, CallExpr):
        arg_form = collect(expr.arg, ctx, depth + 1, verbose)
        if not arg_form.is_constant():
            raise NonLinearError(f"Non-linear: function applied to variable in {expr}")
        fn = KNOWN_FUNCS.get(expr.name)
        if fn is None:
            raise NonLinearError(f"Unknown function: {expr.name}")
        val = fn(arg_form.constant)
        result = LinearForm(constant=val)
        if verbose:
            print(f"{indent}  → Call {expr.name}({arg_form.constant}) = {val} → {result}")
        return result

    raise NonLinearError(f"Unknown node type: {type(expr)}")

## Test Case 1: สมการเชิงเส้นธรรมดา — 2x + 3y - 5

In [ ]:
# 2x + 3y - 5
expr1 = BinaryExpr("+",
    BinaryExpr("+",
        BinaryExpr("*", NumberExpr(2.0), VariableExpr("x")),
        BinaryExpr("*", NumberExpr(3.0), VariableExpr("y")),
    ),
    UnaryExpr("-", NumberExpr(5.0))
)

print(f"Expression: {expr1}")
print()
form1 = collect(expr1, verbose=True)

print(f"\nResult: {form1}")
assert abs(form1.get_coeff("x") - 2.0) < 1e-9
assert abs(form1.get_coeff("y") - 3.0) < 1e-9
assert abs(form1.constant + 5.0) < 1e-9
print("✓ Test 1 passed")

## Test Case 2: Context substitution — x=3 แล้ว 2x + y

เมื่อ context บอกว่า x=3 → 2·3 + y = 6 + y

In [ ]:
# 2x + y, context = {x: 3}
expr2 = BinaryExpr("+",
    BinaryExpr("*", NumberExpr(2.0), VariableExpr("x")),
    VariableExpr("y")
)
ctx2 = {"x": NumberExpr(3.0)}

print(f"Expression: {expr2}")
print(f"Context: x = 3")
print()
form2 = collect(expr2, context=ctx2, verbose=True)

print(f"\nResult: {form2}")
assert form2.is_constant() == False
assert abs(form2.get_coeff("y") - 1.0) < 1e-9
assert abs(form2.constant - 6.0) < 1e-9    # 2·3 = 6
print("✓ Test 2 passed")

## Test Case 3: Function call — sin(π/6) + x

sin(π/6) = 0.5 → constant → ผ่านได้

In [ ]:
# sin(pi/6) + x
expr3 = BinaryExpr("+",
    CallExpr("sin", BinaryExpr("/", VariableExpr("pi"), NumberExpr(6.0))),
    VariableExpr("x")
)
ctx3 = {"pi": NumberExpr(math.pi)}

print(f"Expression: sin(pi/6) + x  (context: pi={math.pi:.4g})")
print()
form3 = collect(expr3, context=ctx3, verbose=True)

print(f"\nResult: {form3}")
assert abs(form3.get_coeff("x") - 1.0) < 1e-9
assert abs(form3.constant - 0.5) < 1e-9   # sin(π/6) = 0.5
print("✓ Test 3 passed: sin(π/6) = 0.5 evaluated as constant")

## Test Case 4: Non-linear — x·y (variable × variable)

คาดหวัง: NonLinearError

In [ ]:
# x * y — ทั้งสอง non-constant
expr4 = BinaryExpr("*", VariableExpr("x"), VariableExpr("y"))

print(f"Expression: {expr4}")
print()
try:
    form4 = collect(expr4, verbose=True)
    print("ERROR: ควร raise NonLinearError!")
except NonLinearError as e:
    print(f"\n✓ NonLinearError raised: {e}")

## Test Case 5: Non-linear — x² (variable^2)

คาดหวัง: NonLinearError

In [ ]:
# x^2
expr5 = BinaryExpr("^", VariableExpr("x"), NumberExpr(2.0))

print(f"Expression: {expr5}")
print()
try:
    form5 = collect(expr5, verbose=True)
    print("ERROR: ควร raise NonLinearError!")
except NonLinearError as e:
    print(f"\n✓ NonLinearError raised: {e}")

## Test Case 6: Non-linear — sin(x) (function ของ variable)

คาดหวัง: NonLinearError

In [ ]:
# sin(x)
expr6 = CallExpr("sin", VariableExpr("x"))

print(f"Expression: {expr6}")
print()
try:
    form6 = collect(expr6, verbose=True)
    print("ERROR: ควร raise NonLinearError!")
except NonLinearError as e:
    print(f"\n✓ NonLinearError raised: {e}")

## Test Case 7: Simplifier — canonicalize equation ax + b = 0

จำลอง `Simplifier::simplify(equation)` ที่สร้าง canonical form

Input: LHS = 3x + y + 2,  RHS = x - y + 1  
→ normalize = LHS - RHS = 2x + 2y + 1 = 0  
→ canonical: "2x + 2y = -1"

In [ ]:
def simplify_equation(
    lhs_expr: Any,
    rhs_expr: Any,
    context: dict | None = None,
    var_order: list[str] | None = None,
    verbose: bool = True,
) -> tuple[LinearForm, str]:
    """
    Simplifier.simplify(equation)

    1. collect LHS → LinearForm
    2. collect RHS → LinearForm
    3. normalize = LHS - RHS
    4. canonical string = "a₁x₁ + a₂x₂ + ... = c" (เอาค่าคงที่ไปขวา)
    """
    if verbose:
        print("=" * 60)
        print(f"Simplify Equation: ({lhs_expr}) = ({rhs_expr})")
        print("=" * 60)
        print("\n--- collect LHS ---")

    lhs_form = collect(lhs_expr, context, verbose=verbose)

    if verbose:
        print(f"\n--- collect RHS ---")
    rhs_form = collect(rhs_expr, context, verbose=verbose)

    # normalize: LHS - RHS → all terms on left, = 0
    norm = (lhs_form - rhs_form).simplify()

    if verbose:
        print(f"\n--- normalize = LHS - RHS ---")
        print(f"  {lhs_form}")
        print(f"- {rhs_form}")
        print(f"= {norm}")

    # Build canonical string: "a₁x₁ + ... = -constant"
    vs = var_order or norm.variables()
    parts = []
    for v in vs:
        c = norm.get_coeff(v)
        if abs(c) < 1e-12:
            continue
        if not parts:
            parts.append(f"{c:g}{v}" if abs(c) != 1 else (v if c > 0 else f"-{v}"))
        else:
            if c > 0:
                parts.append(f"+ {c:g}{v}" if abs(c) != 1 else f"+ {v}")
            else:
                parts.append(f"- {abs(c):g}{v}" if abs(c) != 1 else f"- {v}")

    lhs_str = " ".join(parts) if parts else "0"
    rhs_val = -norm.constant
    canonical = f"{lhs_str} = {rhs_val:g}"

    if verbose:
        print(f"\nCanonical: {canonical}")

    return norm, canonical


# 3x + y + 2 = x - y + 1
lhs7 = BinaryExpr("+",
    BinaryExpr("+",
        BinaryExpr("*", NumberExpr(3.0), VariableExpr("x")),
        VariableExpr("y")
    ),
    NumberExpr(2.0)
)
rhs7 = BinaryExpr("+",
    BinaryExpr("-", VariableExpr("x"), VariableExpr("y")),
    NumberExpr(1.0)
)

norm7, canon7 = simplify_equation(lhs7, rhs7)

assert abs(norm7.get_coeff("x") - 2.0) < 1e-9
assert abs(norm7.get_coeff("y") - 2.0) < 1e-9
assert abs(norm7.constant - 1.0) < 1e-9
print(f"\n✓ Test 7 passed: {canon7}")

## Test Case 8: Recursive context (chain substitution)

context = {a: 2x + 1}  → collect(3a - 1) = 3(2x+1) - 1 = 6x + 2

In [ ]:
# 3a - 1, context = {a: 2x + 1}
expr8 = BinaryExpr("-",
    BinaryExpr("*", NumberExpr(3.0), VariableExpr("a")),
    NumberExpr(1.0)
)
ctx8 = {
    "a": BinaryExpr("+",
        BinaryExpr("*", NumberExpr(2.0), VariableExpr("x")),
        NumberExpr(1.0)
    )
}

print("Expression: 3a - 1")
print("Context: a = 2x + 1")
print("Expected: 6x + 2")
print()
form8 = collect(expr8, context=ctx8, verbose=True)

print(f"\nResult: {form8}")
assert abs(form8.get_coeff("x") - 6.0) < 1e-9
assert abs(form8.constant - 2.0) < 1e-9
print("✓ Test 8 passed: 3(2x+1) - 1 = 6x + 2")

## สรุป Recursion Tree

```
collect(expr):
│
├── NumberExpr(n)      → LinearForm(constant=n)              [Base]
├── VariableExpr(v)    → LinearForm(coeffs={v:1})            [Base]
│                      → collect(context[v])  ถ้ามีใน ctx   [Recursive]
├── UnaryExpr("-")     → collect(child) * -1                 [Recursive × 1]
├── BinaryExpr("+")    → collect(lhs) + collect(rhs)         [Recursive × 2]
├── BinaryExpr("-")    → collect(lhs) - collect(rhs)         [Recursive × 2]
├── BinaryExpr("*")    → ตรวจ linear ก่อน:
│   ├── lhs constant   → collect(rhs) * lhs.constant         [Recursive × 2]
│   ├── rhs constant   → collect(lhs) * rhs.constant         [Recursive × 2]
│   └── ทั้งสอง non-const → ERROR
├── BinaryExpr("/")    → rhs ต้องเป็น constant               [Recursive × 2]
├── BinaryExpr("^")    → exp ต้องเป็น 1 หรือ base constant   [Recursive × 2]
└── CallExpr(fn, arg)  → arg ต้องเป็น constant → fn(val)    [Recursive × 1]
```

| Test | Expression | Context | ผลลัพธ์ |
|------|-----------|---------|--------|
| 1 | 2x+3y-5 | — | coeffs={x:2,y:3}, c=-5 |
| 2 | 2x+y | x=3 | coeffs={y:1}, c=6 |
| 3 | sin(pi/6)+x | pi=π | coeffs={x:1}, c=0.5 |
| 4 | x·y | — | NonLinearError |
| 5 | x² | — | NonLinearError |
| 6 | sin(x) | — | NonLinearError |
| 7 | equation 3x+y+2=x-y+1 | — | canonical: 2x+2y=-1 |
| 8 | 3a-1 | a=2x+1 | coeffs={x:6}, c=2 |